In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
from DFTStructureGenerator import borane_xat_workflow, xtb_process, mol_manipulation
import glob
import os
from rdkit import Chem
import numpy as np
import pandas as pd


In [ ]:
# Gaussian route sections used in the TS-search workflow.
SPE_METHOD = "wb97xd/6-311+g(d,p) scrf=(smd,solvent=toluene) nosymm"
OM_METHOD = "opt=(modredundant,maxcycles=64) freq b3lyp/6-31g(d) em=gd3bj scrf=(smd,solvent=toluene) nosymm"
TS_METHOD = "opt=(calcfc,ts,noeigen,maxcycles=64) freq b3lyp/6-31g(d) em=gd3bj scrf=(smd,solvent=toluene) nosymm"


In [ ]:
# External workspace containing reactant Gaussian/xTB calculation files.
reactant_dir = "E:/work/B_Cl_Nu/Data"


## Load Curated Reactant Tables

These tables provide the optimized B-LB complexes and chloride substrates used to generate TS targets and summarize reaction energies.


In [ ]:
BN_DF_PATH = 'Data/csvs/reactants_B_N.csv'
CL_DF_PATH = 'Data/csvs/reactants_Cl.csv'

BN_df = pd.read_csv(BN_DF_PATH)
Cl_df = pd.read_csv(CL_DF_PATH)


In [ ]:
# Calculation batch name used for generated TS inputs and summaries.
name = "Sum"


In [ ]:
target_dir = os.path.join(r'E:\work\B_Cl_Nu', name)
mol_xtb_file = os.path.join(target_dir, 'OM_XTB')
new_mol_dir = os.path.join(target_dir, 'Mols')
dft_dir_ = os.path.join(target_dir, 'OM')


In [ ]:
duplicate_N_id = [9, 43, 285, 310, 314, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 372, 375, 376]
duplicate_Cl_id = [624, 625, 626, 627, 628, 629, 630, 631, 632, 633, 634, 635, 636,
       637, 638, 639, 640, 642, 644, 645, 652, 653, 654, 655, 656, 657, 658, 659, 660, 661, 662, 663, 664,
       665, 666, 667, 668, 669, 670, 671, 672, 673, 674, 675, 676, 677,
       678, 679, 680, 681, 682, 683, 684, 685, 686, 687, 688, 689, 690,
       691, 692, 693, 694, 695, 696, 697, 698, 699, 700, 701, 702, 703,
       704, 705, 706, 707, 708, 709, 710, 711, 713, 714, 716, 717, 718, 719, 720, 721, 722]

In [ ]:
# Generate constrained xTB inputs for TS guess structures.
if not os.path.isdir(target_dir):
    os.mkdir(target_dir)
if not os.path.isdir(mol_xtb_file):
    os.mkdir(mol_xtb_file)
if not os.path.isdir(new_mol_dir):
    os.mkdir(new_mol_dir)

reaction_csv_path = f'Data/Dataset/{name}.csv'
reaction_csv = pd.read_csv(reaction_csv_path)
all_react_names, all_react_mols, all_restrict = [], [], []
for _, row in reaction_csv.iterrows():
    sum_nwmol, react_name, restrict, _, _ = borane_xat_workflow.generate_ts_structure(
        row,
        2.6,
        2.0,
        target_dir=target_dir,
        reactant_dir=reactant_dir,
        duplicate_N_id=duplicate_N_id,
        duplicate_Cl_id=duplicate_Cl_id,
    )
    if sum_nwmol is None:
        continue
    Chem.MolToMolFile(sum_nwmol, os.path.join(new_mol_dir, f"{react_name}.mol"), kekulize=False)
    all_react_names.append(react_name)
    all_react_mols.append(sum_nwmol)
    all_restrict.append(restrict)

xtb_process.xtb_main(all_react_names, all_react_mols, dir_path=mol_xtb_file, core=50, restrict=all_restrict)


In [ ]:
xtb_process.shift_to_parra(mol_xtb_file, 0, 1)

In [ ]:
# Convert xTB conformers into constrained Gaussian optimizations.
if not os.path.isdir(dft_dir_):
    os.mkdir(dft_dir_)
borane_xat_workflow.smiles_DFT_calc(mol_xtb_file, new_mol_dir, dft_dir_, method=OM_METHOD, conf_limit=3, SpinMultiplicity=2)
for name_, restrict in zip(all_react_names, all_restrict):
    gjfs = glob.glob(os.path.join(dft_dir_, f"{name_}*.gjf"))
    for gjf in gjfs:
        atoms, positions = borane_xat_workflow.FormatConverter.read_gjf(gjf)
        title = " ".join([str(each) for each in [restrict[0][0], restrict[0][1], restrict[1][1]]])
        freeze = restrict[:2]
        borane_xat_workflow.FormatConverter.block_to_gjf(atoms, positions, gjf, 0, 2, title, OM_METHOD, freeze=freeze)


In [ ]:
# Transition-state optimization from constrained-optimization outputs.
borane_xat_workflow.reaction_calc_ts(target_dir, method=TS_METHOD, om_name='OM', ts_name='TS')


In [ ]:
# transition state secondary optimization
bond_attach_function = borane_xat_workflow.logfile_process.bond_addition_function()
bond_attach_function.add_function([0,1], 3.5, 'less')
bond_attach_function.add_function([1,2], 2.5, 'less')
bond_ignore_list = [[0, 1, 2]]
mol_manipulation.error_improve(target_dir, new_mol_dir, 'TS', 'ts_dust_bin', improve_dir='TS_improve', bond_attach_std='mol', bond_addition_function=bond_attach_function, bond_ignore_list = bond_ignore_list, maxcycles=100)

In [ ]:
bond_attach_function = borane_xat_workflow.logfile_process.bond_addition_function()
bond_attach_function.add_function([0,1], 3.5, 'less')
bond_attach_function.add_function([1,2], 2.5, 'less')
bond_ignore_list = [[0, 1, 2]]
mol_manipulation.error_improve(target_dir, new_mol_dir, 'TS_Finish2', 'ts_dust_bin', improve_dir='TS_improve', bond_attach_std='mol', bond_addition_function=bond_attach_function, bond_ignore_list = bond_ignore_list, maxcycles=100)

In [ ]:
borane_xat_workflow.SPE_DFT_calc(target_dir, 'TS', 'TS_ENG', method=SPE_METHOD)

In [ ]:
bond_attach_function = borane_xat_workflow.logfile_process.bond_addition_function()
bond_ignore_list = [[0, 1, 2]]
mol_manipulation.error_improve(target_dir, new_mol_dir, 'TS_ENG', 'TS_ENG_dustbin', improve_dir='TS_ENG_imp', bond_attach_std='mol', bond_addition_function=bond_attach_function, bond_ignore_list = bond_ignore_list, maxcycles=100)


In [ ]:
bond_attach_function = borane_xat_workflow.logfile_process.bond_addition_function()
bond_ignore_list = [[0, 1, 2]]
mol_manipulation.error_improve(target_dir, new_mol_dir, 'TS_ENG', 'TS_ENG_dustbin', improve_dir='TS_ENG_imp', bond_attach_std='mol', bond_addition_function=bond_attach_function, bond_ignore_list = bond_ignore_list, maxcycles=100)
borane_xat_workflow.collection_dft_ts(ts_csv_path=f'Data/Iteration_Data/{name}.csv', dft_dir=target_dir + "/TS", spe_dir=target_dir + "/TS_ENG", duplicate_Cl_id=duplicate_Cl_id)

# IRC Analysis

In [ ]:
all_B_id = BN_df['B_Index'].unique().tolist()
all_N_id = BN_df['N_Index'].unique().tolist()
all_Cl_id = Cl_df['Index'].unique().tolist()

In [ ]:
available_ts_files = []
for each_file in glob.glob(r"E:\work\B_Cl_Nu\Sum\TS_ENG\*.log"):
    B_id, N_id, Cl_id = [int(each) for each in os.path.basename(each_file).split('.')[0].split('_')[1:6:2]]
    if B_id in all_B_id and N_id in all_N_id and Cl_id in all_Cl_id:
        available_ts_files.append([B_id, N_id, Cl_id])
available_ts_files = np.unique(np.array(available_ts_files), axis=0)
print(len(available_ts_files))


In [ ]:
target_dir = r"E:\work\B_Cl_Nu\Sum"
borane_xat_workflow.prepare_irc_jobs(
    available_ts_files=available_ts_files,
    target_dir=target_dir,
    ts_glob_template=r"E:/work/B_Cl_Nu/Sum/TS/B_{B_id:05}_Nu_{N_id:05}_Cl_{Cl_id:05}*.log",
    ts_name="TS",
    ts_need_irc_name="TS_needIRC",
    ts_energy_name="TS_ENG",
    require_all=True,
)


In [ ]:
# Ensure that both IRC endpoints form at least one target bond and cover both directions.
borane_xat_workflow.move_failed_irc_logs(
    target_dir=r"E:\work\B_Cl_Nu\Sum",
    ts_name="TS_needIRC",
    irc_name="IRC_full",
    fail_dir_name="TS_fail_IRC",
    distance_1_max=2.2,
    distance_2_max=2.1,
)


In [ ]:
# Ensure that one IRC endpoint corresponds to B-Cl bonding and the other to C-Cl bonding.
borane_xat_workflow.move_uncertain_irc_logs(
    target_dir=r"E:\work\B_Cl_Nu\Sum",
    ts_name="TS_needIRC",
    irc_name="IRC_full",
    uncertain_dir_name="TS_uncertain_IRC",
    distance_1_bond_max=2.2,
    distance_2_broken_min=2.5,
    distance_1_broken_min=2.6,
    distance_2_bond_max=2.1,
)


# Summary

In [ ]:
BN_energies_r = {f"{row['B_Index']:05}_{row['N_Index']:05}": row['G_energy_r'] for _, row in BN_df.iterrows()}
Cl_energies_r = {f"{row['Index']:05}": row['G_energy_r'] for _, row in Cl_df.iterrows()}
BN_energies_p = {f"{row['B_Index']:05}_{row['N_Index']:05}": row['G_energy_p'] for _, row in BN_df.iterrows()}
Cl_energies_p = {f"{row['Index']:05}": row['G_energy_p'] for _, row in Cl_df.iterrows()}


In [ ]:
borane_xat_workflow.export_ts_summary_csv(
    target_dir=r"E:\work\B_Cl_Nu\Sum",
    bn_energies_r=BN_energies_r,
    cl_energies_r=Cl_energies_r,
    bn_energies_p=BN_energies_p,
    cl_energies_p=Cl_energies_p,
    output_csv=r"E:/work/B_Cl_Nu/Sum/result.csv",
    ts_name="TS_needIRC",
    ts_eng_name="TS_ENG",
)


In [ ]:
borane_xat_workflow.annotate_reaction_aam_csv(
    target_csv_path=r"E:/work/B_Cl_Nu/Sum/result_clear.csv",
    output_csv_path=r"E:/work/B_Cl_Nu/Sum/result_clear_AAM.csv",
)


## Optional: Complete a TS Target CSV

This utility fills a compact target list containing `B_Index`, `N_Index`, and `Cl_Index` with the SMILES, reactive atom ids, conformer ids, and reactant energies required by the TS workflow. It is kept at the end because it is a table-repair step, not part of the main TS search loop.


In [ ]:
borane_xat_workflow.complete_target(
    pd.read_csv(r"E:/work/B_Cl_Nu/Sum/result_clear.csv"),
    bn_df=BN_df,
    cl_df=Cl_df,
    name='result_clear',
)
